# TEST SINGLE AUGMENTATION

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import torch
from typing import Any

from emma import EmmaDataset, EmmaSearcher, SearchQuery
from transformers import AutoTokenizer, CLIPTextModelWithProjection

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"])
asins = se.query_asins(query)

In [ ]:
# SELECT A SAMPLE
asin = asins[10]
rec = ds.get_record(asin=asin)
title = ds.get_title(rec=rec)
categories = ds.get_category_names(rec=rec)

print("Title: ", title)
print("Category Path: ", categories)

## > MODEL

In [ ]:
clip_root = None  # e.g. Path("/path/to/CLIP")

if clip_root is None:
    raise ValueError("Please set `clip_root` to the path of the CLIP model.")

clip_root = Path(clip_root)

if not clip_root.exists():
    raise FileNotFoundError(f"Model path does not exist: {clip_root}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(clip_root)
model = CLIPTextModelWithProjection.from_pretrained(
    clip_root,
    use_safetensors=True,
).to(device)
model.eval()
model.requires_grad_(False)
print(f"CLIP text encoder loaded on {device}")

## > AUGMENTATION

In [ ]:
def flatten_category_hierarchy(categories: Any) -> list[str]:
    if isinstance(categories, str):
        value = categories.strip()
        return [value]
    if isinstance(categories, (list, tuple)):
        hierarchy: list[str] = []
        for category in categories:
            hierarchy.extend(flatten_category_hierarchy(category))
        return hierarchy

    value = str(categories).strip()
    return [value]


@torch.inference_mode()
def encode_texts(
    model: CLIPTextModelWithProjection,
    tokenizer: Any,
    device: torch.device,
    texts: list[str],
) -> torch.Tensor:
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=model.config.max_position_embeddings,
        return_tensors="pt",
    ).to(device)

    embeddings = model(**inputs).text_embeds

    return embeddings.float().cpu()

In [ ]:
hierarchy = flatten_category_hierarchy(categories)
text = " > ".join(hierarchy)

print("INPUT: ", text)

In [ ]:
embeddings = encode_texts(model, tokenizer, device, text).numpy()

print(np.shape(embeddings))
print(np.mean(embeddings))